In [1]:
import os
from pyspark.sql import SparkSession

# 1. AWS 자격증명 프로필 설정 (터미널에서 -e AWS_PROFILE=metacode 한 것과 동일한 효과)
os.environ["AWS_PROFILE"] = "metacode"

# 2. Spark Session 생성 및 Iceberg/Glue 환경 세팅
spark = (
    SparkSession.builder.appName("Iceberg-Jupyter")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
    .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.glue_catalog.warehouse", "s3a://metacode-iceberg-0426/warehouse")
    .getOrCreate()
)

print("Spark Iceberg 환경 세팅 완료!")

Spark Iceberg 환경 세팅 완료!


In [4]:
# Silver 테이블 데이터 5줄 확인
spark.sql("SELECT * FROM glue_catalog.ad_lakehouse.processed_events LIMIT 5").show()

# 파티션(날짜)별 데이터 건수 확인
spark.sql("SELECT event_date, COUNT(*) as cnt FROM glue_catalog.ad_lakehouse.processed_events GROUP BY event_date ORDER BY event_date DESC").show()

+------------+----------+--------+--------+-----+----------+--------------------+----------------+--------------------+
|    event_id|event_date|     uid|campaign|click|conversion|conversion_delay_sec|            cost|          updated_at|
+------------+----------+--------+--------+-----+----------+--------------------+----------------+--------------------+
|evt_00000712|2026-04-01|18750238|25920690|    0|         0|                NULL|8.37021415681E-4|2026-04-30 11:11:...|
|evt_00000713|2026-04-01|11410280|15654890|    0|         0|                NULL|          1.0E-5|2026-04-30 11:11:...|
|evt_00000714|2026-04-01| 6507341|15885288|    0|         0|                NULL|4.77762898131E-4|2026-04-30 11:11:...|
|evt_00000715|2026-04-01|11477114|17288262|    0|         0|                NULL|3.97058830541E-4|2026-04-30 11:11:...|
|evt_00000716|2026-04-01|14240056|31772643|    1|         0|                NULL|3.74999992988E-5|2026-04-30 11:11:...|
+------------+----------+--------+------